# M2 - Synthetic Fab-Style Production Line: Generator Demo

This notebook demonstrates the synthetic production-line generator and the two
validation checks that make it defensible:

1. **Little's Law self-consistency** - measured WIP must match throughput x cycle time.
2. **Ground-truth bottleneck recovery** - the engineered station (S4) must show the
   highest empirical utilization.

The data is **synthetic and clearly labeled**. Its purpose is to demonstrate
methodology with a known ground truth, not to predict any real fab. Structure
(re-entrant route, tool groups) is informed by the public SMT2020 testbed.

In [ ]:
import sys
sys.path.append("../src/generator")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from factory_generator import default_config, simulate, theoretical_utilization

cfg = default_config()
log, lifecycle, meta = simulate(cfg)
log.head()

## Event log schema

One row per completed operation. `case = lot_id`, `activity = station`, with
queue / start / complete timestamps - directly usable for process mining,
cycle-time, WIP and utilization analysis.

In [ ]:
print("Operations:", len(log), "| Lots:", len(lifecycle))
print("Route:", meta["route"])
print("Designed bottleneck:", meta["ground_truth_bottleneck"])
log.describe(include="all").T

## Validation 1 - Little's Law

In [ ]:
t0, t1 = cfg.warmup_hours, cfg.horizon_hours
window = t1 - t0

done = lifecycle.dropna(subset=["completion_time"])
in_win = done[(done.completion_time >= t0) & (done.completion_time <= t1)].copy()
in_win["cycle_time"] = in_win.completion_time - in_win.arrival_time

throughput = len(in_win) / window
avg_ct = in_win.cycle_time.mean()
wip_ll = throughput * avg_ct

arr = lifecycle.arrival_time.to_numpy()
comp = lifecycle.completion_time.fillna(t1).to_numpy()
overlap = np.clip(np.minimum(comp, t1) - np.maximum(arr, t0), 0, None)
wip_meas = overlap.sum() / window

print(f"Throughput (lots/h) : {throughput:.3f}")
print(f"Avg cycle time (h)  : {avg_ct:.2f}")
print(f"WIP predicted       : {wip_ll:.2f}")
print(f"WIP measured        : {wip_meas:.2f}")
print(f"Gap                 : {abs(wip_meas-wip_ll)/wip_meas*100:.2f}%")

## Validation 2 - Bottleneck recovery

In [ ]:
theo = theoretical_utilization(cfg)
emp = {}
for s, st in cfg.stations.items():
    ops = log[log.station == s]
    start = ops.process_start_time.clip(lower=t0, upper=t1)
    end = ops.process_complete_time.clip(lower=t0, upper=t1)
    emp[s] = (end - start).clip(lower=0).sum() / (st.n_tools * window)

util = pd.DataFrame({"planned": theo, "observed": emp})
ax = util.plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("Utilization")
ax.set_title("Station utilization: planned vs observed (S4 = bottleneck)")
plt.tight_layout(); plt.show()
print("Empirical bottleneck:", max(emp, key=emp.get))

## Cycle-time distribution & WIP over time

These two views are the seed for the M3 KPI dashboard.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

in_win.cycle_time.hist(bins=40, ax=ax[0])
ax[0].set_title("Cycle-time distribution (hours)")
ax[0].set_xlabel("cycle time")

# WIP(t) via a step function from arrivals (+1) and completions (-1)
events = [(a, 1) for a in lifecycle.arrival_time]
events += [(c, -1) for c in lifecycle.completion_time.dropna()]
events.sort()
times = [e[0] for e in events]
wip = np.cumsum([e[1] for e in events])
ax[1].plot(times, wip, lw=0.8)
ax[1].axvline(t0, color="r", ls="--", lw=1, label="end of warm-up")
ax[1].set_title("WIP over time"); ax[1].set_xlabel("hours"); ax[1].legend()
plt.tight_layout(); plt.show()